In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as Data
import numpy as np
from torch import optim
from warnings import warn
import pandas as pd

import gc
import hyperopt
from hyperopt import hp, fmin, tpe, Trials, partial
from hyperopt.early_stop import no_progress_loss
import contextlib

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from collections import OrderedDict
from sklearn.metrics import r2_score, mean_squared_error,explained_variance_score,mean_absolute_error
from torch.jit import script

In [2]:
def to_one_hot(y, depth=None):
    r"""
    Takes integer with n dims and converts it to 1-hot representation with n + 1 dims.
    The n+1'st dimension will have zeros everywhere but at y'th index, where it will be equal to 1.
    Args:
        y: input integer (IntTensor, LongTensor or Variable) of any shape
        depth (int):  the size of the one hot dimension
    """
    y_flat = y.to(torch.int64).view(-1, 1)
    depth = depth if depth is not None else int(torch.max(y_flat)) + 1
    y_one_hot = torch.zeros(y_flat.size()[0], depth, device=y.device).scatter_(1, y_flat, 1)
    y_one_hot = y_one_hot.view(*(tuple(y.shape) + (-1,)))
    return y_one_hot


def _make_ix_like(input, dim=0):
    d = input.size(dim)
    rho = torch.arange(1, d + 1, device=input.device, dtype=input.dtype)
    view = [1] * input.dim()
    view[0] = -1
    return rho.view(view).transpose(0, dim)


class SparsemaxFunction(Function):
    """
    An implementation of sparsemax (Martins & Astudillo, 2016). See
    :cite:`DBLP:journals/corr/MartinsA16` for detailed description.

    By Ben Peters and Vlad Niculae
    """

    @staticmethod
    def forward(ctx, input, dim=-1):
        """sparsemax: normalizing sparse transform (a la softmax)

        Parameters:
            input (Tensor): any shape
            dim: dimension along which to apply sparsemax

        Returns:
            output (Tensor): same shape as input
        """
        ctx.dim = dim
        max_val, _ = input.max(dim=dim, keepdim=True)
        input -= max_val  # same numerical stability trick as for softmax
        tau, supp_size = SparsemaxFunction._threshold_and_support(input, dim=dim)
        output = torch.clamp(input - tau, min=0)
        ctx.save_for_backward(supp_size, output)
        return output

    @staticmethod
    def backward(ctx, grad_output):
        supp_size, output = ctx.saved_tensors
        dim = ctx.dim
        grad_input = grad_output.clone()
        grad_input[output == 0] = 0

        v_hat = grad_input.sum(dim=dim) / supp_size.to(output.dtype).squeeze()
        v_hat = v_hat.unsqueeze(dim)
        grad_input = torch.where(output != 0, grad_input - v_hat, grad_input)
        return grad_input, None


    @staticmethod
    def _threshold_and_support(input, dim=-1):
        """Sparsemax building block: compute the threshold

        Args:
            input: any dimension
            dim: dimension along which to apply the sparsemax

        Returns:
            the threshold value
        """

        input_srt, _ = torch.sort(input, descending=True, dim=dim)
        input_cumsum = input_srt.cumsum(dim) - 1
        rhos = _make_ix_like(input, dim)
        support = rhos * input_srt > input_cumsum

        support_size = support.sum(dim=dim).unsqueeze(dim)
        tau = input_cumsum.gather(dim, support_size - 1)
        tau /= support_size.to(input.dtype)
        return tau, support_size


sparsemax = lambda input, dim=-1: SparsemaxFunction.apply(input, dim)
sparsemoid = lambda input: (0.5 * input + 0.5).clamp_(0, 1)


class Entmax15Function(Function):
    """
    An implementation of exact Entmax with alpha=1.5 (B. Peters, V. Niculae, A. Martins). See
    :cite:`https://arxiv.org/abs/1905.05702 for detailed description.
    Source: https://github.com/deep-spin/entmax
    """

    @staticmethod
    def forward(ctx, input, dim=-1):
        ctx.dim = dim

        max_val, _ = input.max(dim=dim, keepdim=True)
        input = input - max_val  # same numerical stability trick as for softmax
        input = input / 2  # divide by 2 to solve actual Entmax

        tau_star, _ = Entmax15Function._threshold_and_support(input, dim)
        output = torch.clamp(input - tau_star, min=0) ** 2
        ctx.save_for_backward(output)
        return output

    @staticmethod
    def backward(ctx, grad_output):
        Y, = ctx.saved_tensors
        gppr = Y.sqrt()  # = 1 / g'' (Y)
        dX = grad_output * gppr
        q = dX.sum(ctx.dim) / gppr.sum(ctx.dim)
        q = q.unsqueeze(ctx.dim)
        dX -= q * gppr
        return dX, None

    @staticmethod
    def _threshold_and_support(input, dim=-1):
        Xsrt, _ = torch.sort(input, descending=True, dim=dim)

        rho = _make_ix_like(input, dim)
        mean = Xsrt.cumsum(dim) / rho
        mean_sq = (Xsrt ** 2).cumsum(dim) / rho
        ss = rho * (mean_sq - mean ** 2)
        delta = (1 - ss) / rho

        # NOTE this is not exactly the same as in reference algo
        # Fortunately it seems the clamped values never wrongly
        # get selected by tau <= sorted_z. Prove this!
        delta_nz = torch.clamp(delta, 0)
        tau = mean - torch.sqrt(delta_nz)

        support_size = (tau <= Xsrt).sum(dim).unsqueeze(dim)
        tau_star = tau.gather(dim, support_size - 1)
        return tau_star, support_size


class Entmoid15(Function):
    """ A highly optimized equivalent of labda x: Entmax15([x, 0]) """

    @staticmethod
    def forward(ctx, input):
        output = Entmoid15._forward(input)
        ctx.save_for_backward(output)
        return output

    @staticmethod
    @script
    def _forward(input):
        input, is_pos = abs(input), input >= 0
        tau = (input + torch.sqrt(F.relu(8 - input ** 2))) / 2
        tau.masked_fill_(tau <= input, 2.0)
        y_neg = 0.25 * F.relu(tau - input, inplace=True) ** 2
        return torch.where(is_pos, 1 - y_neg, y_neg)

    @staticmethod
    def backward(ctx, grad_output):
        return Entmoid15._backward(ctx.saved_tensors[0], grad_output)

    @staticmethod
    @script
    def _backward(output, grad_output):
        gppr0, gppr1 = output.sqrt(), (1 - output).sqrt()
        grad_input = grad_output * gppr0
        q = grad_input / (gppr0 + gppr1)
        grad_input -= q * gppr0
        return grad_input


entmax15 = lambda input, dim=-1: Entmax15Function.apply(input, dim)
entmoid15 = Entmoid15.apply


class Lambda(nn.Module):
    def __init__(self, func):
        super().__init__()
        self.func = func

    def forward(self, *args, **kwargs):
        return self.func(*args, **kwargs)


class ModuleWithInit(nn.Module):
    """ Base class for pytorch module with data-aware initializer on first batch """
    def __init__(self):
        super().__init__()
        self._is_initialized_tensor = nn.Parameter(torch.tensor(0, dtype=torch.uint8), requires_grad=False)
        self._is_initialized_bool = None
        # Note: this module uses a separate flag self._is_initialized so as to achieve both
        # * persistence: is_initialized is saved alongside model in state_dict
        # * speed: model doesn't need to cache
        # please DO NOT use these flags in child modules

    def initialize(self, *args, **kwargs):
        """ initialize module tensors using first batch of data """
        raise NotImplementedError("Please implement ")

    def __call__(self, *args, **kwargs):
        if self._is_initialized_bool is None:
            self._is_initialized_bool = bool(self._is_initialized_tensor.item())
        if not self._is_initialized_bool:
            self.initialize(*args, **kwargs)
            self._is_initialized_tensor.data[...] = 1
            self._is_initialized_bool = True
        return super().__call__(*args, **kwargs)
def check_numpy(x):
    """ Makes sure x is a numpy array """
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    assert isinstance(x, np.ndarray)
    return x

In [3]:
class ODST(ModuleWithInit):
    def __init__(self, in_features, num_trees, depth=6, tree_dim=2, flatten_output=True,
                 choice_function=sparsemax, bin_function=sparsemoid,
                 initialize_response_=nn.init.normal_, initialize_selection_logits_=nn.init.uniform_,
                 threshold_init_beta=1.0, threshold_init_cutoff=1.0,
                 ):
        """
        Oblivious Differentiable Sparsemax Trees. http://tinyurl.com/odst-readmore
        One can drop (sic!) this module anywhere instead of nn.Linear
        :param in_features: number of features in the input tensor
        :param num_trees: number of trees in this layer
        :param tree_dim: number of response channels in the response of individual tree
        :param depth: number of splits in every tree
        :param flatten_output: if False, returns [..., num_trees, tree_dim],
            by default returns [..., num_trees * tree_dim]
        :param choice_function: f(tensor, dim) -> R_simplex computes feature weights s.t. f(tensor, dim).sum(dim) == 1
        :param bin_function: f(tensor) -> R[0, 1], computes tree leaf weights

        :param initialize_response_: in-place initializer for tree output tensor
        :param initialize_selection_logits_: in-place initializer for logits that select features for the tree
        both thresholds and scales are initialized with data-aware init (or .load_state_dict)
        :param threshold_init_beta: initializes threshold to a q-th quantile of data points
            where q ~ Beta(:threshold_init_beta:, :threshold_init_beta:)
            If this param is set to 1, initial thresholds will have the same distribution as data points
            If greater than 1 (e.g. 10), thresholds will be closer to median data value
            If less than 1 (e.g. 0.1), thresholds will approach min/max data values.

        :param threshold_init_cutoff: threshold log-temperatures initializer, \in (0, inf)
            By default(1.0), log-remperatures are initialized in such a way that all bin selectors
            end up in the linear region of sparse-sigmoid. The temperatures are then scaled by this parameter.
            Setting this value > 1.0 will result in some margin between data points and sparse-sigmoid cutoff value
            Setting this value < 1.0 will cause (1 - value) part of data points to end up in flat sparse-sigmoid region
            For instance, threshold_init_cutoff = 0.9 will set 10% points equal to 0.0 or 1.0
            Setting this value > 1.0 will result in a margin between data points and sparse-sigmoid cutoff value
            All points will be between (0.5 - 0.5 / threshold_init_cutoff) and (0.5 + 0.5 / threshold_init_cutoff)
        """
        super().__init__()
        self.depth, self.num_trees, self.tree_dim, self.flatten_output = depth, num_trees, tree_dim, flatten_output
        self.choice_function, self.bin_function = choice_function, bin_function
        self.threshold_init_beta, self.threshold_init_cutoff = threshold_init_beta, threshold_init_cutoff

        self.response = nn.Parameter(torch.zeros([num_trees, tree_dim, 2 ** depth]), requires_grad=True)
        initialize_response_(self.response)

        self.feature_selection_logits = nn.Parameter(
            torch.zeros([in_features, num_trees, depth]), requires_grad=True
        )
        initialize_selection_logits_(self.feature_selection_logits)

        self.feature_thresholds = nn.Parameter(
            torch.full([num_trees, depth], float('nan'), dtype=torch.float32), requires_grad=True
        )  # nan values will be initialized on first batch (data-aware init)

        self.log_temperatures = nn.Parameter(
            torch.full([num_trees, depth], float('nan'), dtype=torch.float32), requires_grad=True
        )
        self.fc = nn.Linear(num_trees * tree_dim, tree_dim)
        # binary codes for mapping between 1-hot vectors and bin indices
        with torch.no_grad():
            indices = torch.arange(2 ** self.depth)
            offsets = 2 ** torch.arange(self.depth)
            bin_codes = (indices.view(1, -1) // offsets.view(-1, 1) % 2).to(torch.float32)
            bin_codes_1hot = torch.stack([bin_codes, 1.0 - bin_codes], dim=-1)
            self.bin_codes_1hot = nn.Parameter(bin_codes_1hot, requires_grad=False)
            # ^-- [depth, 2 ** depth, 2]

    def forward(self, input):
        assert len(input.shape) >= 2
        if len(input.shape) > 2:
            return self.forward(input.view(-1, input.shape[-1])).view(*input.shape[:-1], -1)
        # new input shape: [batch_size, in_features]

        feature_logits = self.feature_selection_logits
        feature_selectors = self.choice_function(feature_logits, dim=0)
        # ^--[in_features, num_trees, depth]

        feature_values = torch.einsum('bi,ind->bnd', input, feature_selectors)
        # ^--[batch_size, num_trees, depth]

        threshold_logits = (feature_values - self.feature_thresholds) * torch.exp(-self.log_temperatures)

        threshold_logits = torch.stack([-threshold_logits, threshold_logits], dim=-1)
        # ^--[batch_size, num_trees, depth, 2]

        bins = self.bin_function(threshold_logits)
        # ^--[batch_size, num_trees, depth, 2], approximately binary

        bin_matches = torch.einsum('btds,dcs->btdc', bins, self.bin_codes_1hot)
        # ^--[batch_size, num_trees, depth, 2 ** depth]

        response_weights = torch.prod(bin_matches, dim=-2)
        # ^-- [batch_size, num_trees, 2 ** depth]

        response = torch.einsum('bnd,ncd->bnc', response_weights, self.response)
        # ^-- [batch_size, num_trees, tree_dim]

        return self.fc(response.flatten(1, 2) if self.flatten_output else response)

    def initialize(self, input, eps=1e-6):
        # data-aware initializer
        assert len(input.shape) == 2
#         if input.shape[0] < 1000:
#             warn("Data-aware initialization is performed on less than 1000 data points. This may cause instability."
#                  "To avoid potential problems, run this model on a data batch with at least 1000 data samples."
#                  "You can do so manually before training. Use with torch.no_grad() for memory efficiency.")
        with torch.no_grad():
            feature_selectors = self.choice_function(self.feature_selection_logits, dim=0)
            # ^--[in_features, num_trees, depth]

            feature_values = torch.einsum('bi,ind->bnd', input, feature_selectors)
            # ^--[batch_size, num_trees, depth]

            # initialize thresholds: sample random percentiles of data
            percentiles_q = 100 * np.random.beta(self.threshold_init_beta, self.threshold_init_beta,
                                                 size=[self.num_trees, self.depth])
            self.feature_thresholds.data[...] = torch.as_tensor(
                list(map(np.percentile, check_numpy(feature_values.flatten(1, 2).t()), percentiles_q.flatten())),
                dtype=feature_values.dtype, device=feature_values.device
            ).view(self.num_trees, self.depth)

            # init temperatures: make sure enough data points are in the linear region of sparse-sigmoid
            temperatures = np.percentile(check_numpy(abs(feature_values - self.feature_thresholds)),
                                         q=100 * min(1.0, self.threshold_init_cutoff), axis=0)

            # if threshold_init_cutoff > 1, scale everything down by it
            temperatures /= max(1.0, self.threshold_init_cutoff)
            self.log_temperatures.data[...] = torch.log(torch.as_tensor(temperatures) + eps)

    def __repr__(self):
        return "{}(in_features={}, num_trees={}, depth={}, tree_dim={}, flatten_output={})".format(
            self.__class__.__name__, self.feature_selection_logits.shape[0],
            self.num_trees, self.depth, self.tree_dim, self.flatten_output
        )


In [4]:
df = pd.read_csv('micedata.csv')
df

,0.73809416178386,-0.0239958526540536,1,1.1,1.2,1.3,0,1.4,0.1,1.5,...,0.6313,0.6314,0.6315,0.6316,0.6317,0.6318,0.6319,0.6320,1.4023,1.4024
0,0.743638,0.037044,1,1,2,1,1,1,1,2,...,0,0,0,0,0,0,0,0,2,0
1,0.431086,-0.082434,2,0,2,2,0,2,0,2,...,0,0,0,0,0,0,0,0,2,2
2,0.144433,0.017911,1,1,1,1,1,1,1,2,...,0,0,0,0,0,0,0,0,2,0
3,0.675366,-0.040148,2,0,2,2,0,2,0,2,...,1,1,1,1,1,1,1,1,1,1
4,-0.658133,0.043124,1,1,1,1,0,1,0,1,...,0,0,0,0,0,0,0,0,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1808,-0.157640,0.021284,1,1,1,1,0,1,0,1,...,0,0,0,0,0,0,0,0,2,2
1809,0.751668,-0.073322,1,1,2,1,1,1,1,2,...,0,0,0,0,0,0,0,0,2,2
1810,0.186213,0.056111,1,1,2,1,1,1,1,2,...,0,0,0,0,0,0,0,0,2,2
1811,-0.779898,0.020232,1,1,1,1,1,1,1,2,...,0,0,0,1,0,0,0,2,2,0


In [5]:
def z_score(data):
    data = data.astype(float)
    Mean = data.mean()
    Var = ((data - Mean)**2).mean()
    Std = pow(Var,0.5)
    data = (data - Mean)/Std  # 标准化
    return Mean,Std,data
Mean,Std,df.iloc[:,:2] = z_score(df.iloc[:,:2])

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, Y_train, Y_test = train_test_split(df.iloc[:,2:].values, df.iloc[:,:2].values,test_size=0.2,random_state=42)
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)

(1450, 10346) (363, 10346) (1450, 2) (363, 2)


In [8]:
Y_train = np.squeeze(Y_train)
Y_test = np.squeeze(Y_test)


X_train = torch.tensor(X_train,dtype = torch.float)
Y_train  = torch.tensor(Y_train,dtype = torch.float)

X_test = torch.tensor(X_test,dtype = torch.float)
Y_test  = torch.tensor(Y_test,dtype = torch.float)



print(X_train.shape, Y_train.shape,X_test.shape, Y_test.shape)

torch.Size([1450, 10346]) torch.Size([1450, 2]) torch.Size([363, 10346]) torch.Size([363, 2])


In [9]:
train_loader = Data.DataLoader(
    dataset=Data.TensorDataset(X_train, Y_train),  # 封装进Data.TensorDataset()类的数据，可以为任意维度
    batch_size=15,  # 每块的大小
    shuffle=True,  # 要不要打乱数据 (打乱比较好)
    drop_last =True, #丢弃最后一组数据
    num_workers=0,  # 多进程（multiprocess）来读数据
)
test_loader = Data.DataLoader(
    dataset=Data.TensorDataset(X_test, Y_test),  # 封装进Data.TensorDataset()类的数据，可以为任意维度
    batch_size=15,  # 每块的大小
    shuffle=True,  # 要不要打乱数据 (打乱比较好)
    drop_last =True, 
    num_workers=0,  # 多进程（multiprocess）来读数据
)

In [10]:
import time
def train(params):
#     print(params)
    start_time = time.time()


    lr=params['lr']
    step_size=params['step_size']
    gamma=params['gamma']
    num_trees=params['num_trees']
    threshold_init_beta=params['threshold_init_beta']
    threshold_init_cutoff=params['threshold_init_cutoff']

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = ODST(in_features=df.shape[1]-2, num_trees=num_trees,threshold_init_beta=threshold_init_beta,threshold_init_cutoff=threshold_init_cutoff)
    model = model.to(device)
    loss_function = nn.MSELoss()  # loss
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # 优化器
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    Epochs = 50
    model.train()
    loss_list = []
    for epochs in range(Epochs):
        model.train()
        r2_test =0
        train_loss = 0
        for data_l in train_loader:
            seq, labels = data_l
            seq, labels = seq.to(device), labels.to(device)
            optimizer.zero_grad()                          #    清空过往梯度 
            y_pred = model(seq)
            labels = torch.squeeze(labels)
            single_loss = loss_function(y_pred, labels)    #    获取loss：输入预测值和标签，计算损失函数
            single_loss.backward()                         #    反向传播，计算当前梯度
            optimizer.step()                               #    根据梯度更新网络参数
            train_loss += single_loss.item()
        scheduler.step()
        model.eval()
        y_pred = model(X_test.to(device))
        test_loss =loss_function(y_pred, torch.squeeze(Y_test).to(device))
        loss_list.append(test_loss.item())
        del seq, labels ,y_pred #删除数据与变量
        gc.collect() #清除数据与变量相关的缓存
        torch.cuda.empty_cache() #缓存分配器分配出去的内存给释放掉
#     print(np.min(loss_list))

    end_time = time.time()

    # 计算运行时间
    run_time = end_time - start_time
    print(f"程序运行时间：{run_time} 秒")
    return np.min(loss_list)

In [11]:
param_grid_simple = {'lr': hp.uniform("lr",0.0001,0.3)
                     ,'step_size':hp.randint('step_size',99)+1
                     ,'gamma':hp.uniform("gamma",0.1,1)
                     ,'num_trees':hp.randint('num_trees',95)+5
                     ,'threshold_init_beta':hp.uniform("threshold_init_beta",0.1,10)
                     ,'threshold_init_cutoff':hp.uniform("threshold_init_cutoff",0.1,10)
                    }

In [12]:
def param_hyperopt(max_evals=100):
    
    #保存迭代过程
    trials = Trials()
    
    #设置提前停止
    early_stop_fn = no_progress_loss(100)
    
    #定义代理模型
    #algo = partial(tpe.suggest, n_startup_jobs=20, n_EI_candidates=50)
    params_best = fmin(train #目标函数
                       , space = param_grid_simple #参数空间
                       , algo = tpe.suggest 
                       #, algo = algo
                       , max_evals = max_evals #允许的迭代次数
                       , verbose=True
                       , trials = trials
                       , early_stop_fn = early_stop_fn
                      )
    
    #打印最优参数，fmin会自动打印最佳分数
    print("\n","\n","best params: ", params_best,
          "\n")
    return params_best, trials
param_hyperopt(max_evals=100)

程序运行时间：27.755406618118286 秒                            
程序运行时间：21.822744131088257 秒                                                      
程序运行时间：19.07805633544922 秒                                                       
程序运行时间：20.458975315093994 秒                                                      
程序运行时间：21.171698331832886 秒                                                      
程序运行时间：21.6705060005188 秒                                                        
程序运行时间：30.51681876182556 秒                                                       
程序运行时间：20.302414655685425 秒                                                      
程序运行时间：24.770138025283813 秒                                                      
程序运行时间：21.311951160430908 秒                                                      
程序运行时间：30.89328122138977 秒                                                        
程序运行时间：32.5052535533905 秒                                                         
程序运行时间：31.186708450317383 秒             

({'gamma': 0.21122084967870858,
  'lr': 0.04775286650108621,
  'num_trees': 17,
  'step_size': 66,
  'threshold_init_beta': 3.4777399934272917,
  'threshold_init_cutoff': 1.4101645802718543},
 <hyperopt.base.Trials at 0x7f84db2c5f10>)

In [13]:

mse2 = []
rmse2 = []
mae2 = []
mape2 = []
params = {'gamma': 0.21122084967870858,
  'lr': 0.04775286650108621,
  'num_trees': 17,
  'step_size': 66,
  'threshold_init_beta': 3.4777399934272917,
  'threshold_init_cutoff': 1.4101645802718543}
lr=params['lr']
step_size=params['step_size']
gamma=params['gamma']
num_trees=params['num_trees']
threshold_init_beta=params['threshold_init_beta']
threshold_init_cutoff=params['threshold_init_cutoff']
for cv in range(10):
    print('交叉验证第{}折'.format(cv+1))
    X_train, X_test, Y_train, Y_test = train_test_split(df.iloc[:,2:].values, df.iloc[:,:2].values,test_size=0.2,random_state=cv)
    Y_train = np.squeeze(Y_train)
    Y_test = np.squeeze(Y_test)
    X_train = torch.tensor(X_train,dtype = torch.float)
    Y_train  = torch.tensor(Y_train,dtype = torch.float)
    X_test = torch.tensor(X_test,dtype = torch.float)
    Y_test  = torch.tensor(Y_test,dtype = torch.float)
    print(X_train.shape, Y_train.shape,X_test.shape, Y_test.shape)
    train_loader = Data.DataLoader(
        dataset=Data.TensorDataset(X_train, Y_train),  # 封装进Data.TensorDataset()类的数据，可以为任意维度
        batch_size=15,  # 每块的大小
        shuffle=True,  # 要不要打乱数据 (打乱比较好)
        drop_last =True, #丢弃最后一组数据
        num_workers=0,  # 多进程（multiprocess）来读数据
    )
    test_loader = Data.DataLoader(
        dataset=Data.TensorDataset(X_test, Y_test),  # 封装进Data.TensorDataset()类的数据，可以为任意维度
        batch_size=15,  # 每块的大小
        shuffle=True,  # 要不要打乱数据 (打乱比较好)
        drop_last =True, 
        num_workers=0,  # 多进程（multiprocess）来读数据
    )
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = ODST(in_features=df.shape[1]-2, num_trees=num_trees,threshold_init_beta=threshold_init_beta,threshold_init_cutoff=threshold_init_cutoff)
    model = model.to(device)
    loss_function = nn.MSELoss()  # loss
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # 优化器
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    Epochs = 50
    model.train()
    RR2 = -100
    result = {}
    result['train-loss']= []
    result['test-loss']= []
    mse1 = []
    rmse1 = []
    mae1 = []
    mape1 = []
    for epochs in range(Epochs):
        model.train()
        r2_test =0
        train_loss = 0
        for data_l in train_loader:
            seq, labels = data_l
            seq, labels = seq.to(device), labels.to(device)
            optimizer.zero_grad()                          #    清空过往梯度 
            y_pred = model(seq)
            labels = torch.squeeze(labels)
            single_loss = loss_function(y_pred, labels)    #    获取loss：输入预测值和标签，计算损失函数
            single_loss.backward()                         #    反向传播，计算当前梯度
            optimizer.step()                               #    根据梯度更新网络参数
            train_loss += single_loss.item()
        scheduler.step()
        model.eval()
        y_pred = model(X_test.to(device))
        test_loss =loss_function(y_pred, torch.squeeze(Y_test).to(device))
        r2_test = r2_score(np.squeeze(Y_test.cpu().detach().numpy()),y_pred.cpu().detach().numpy())
        if RR2 < r2_test:
            RR2 =r2_test
            # torch.save(model, 'model_NODE1.pth')
            torch.save(model.state_dict(), 'model_NODE1.pth')
            # print('已更新保存模型')
        result['train-loss'].append(train_loss/len(train_loader))
        result['test-loss'].append(test_loss.item())
        # print('Epochs',epochs,'loss_train',train_loss/len(train_loader),'loss_test',test_loss.item(),'r2_test',r2_test,)
        del seq, labels ,y_pred #删除数据与变量
        gc.collect() #清除数据与变量相关的缓存
        torch.cuda.empty_cache() #缓存分配器分配出去的内存给释放掉
    model.load_state_dict(torch.load('model_NODE1.pth'))
    pred = model(X_test.to(device))
    pred = pred.cpu().detach().numpy()
    Y_test1 = Y_test.cpu().detach().numpy()
    for i in range(2):
        Y_test1 = Y_test.cpu().detach().numpy()[:,i]
        pre     = pred[:,i]
        mse = mean_squared_error(Y_test1,pre)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(Y_test1,pre)

        pre = pre.reshape(-1)
        Y_test1 = Y_test1.reshape(-1)
        INDEX = []
        page = 0
        for i in Y_test1:
            if i ==0:
                INDEX.append(page)
            page +=1
        if INDEX !=[]:
            Y_test1 = np.delete(Y_test1,INDEX,0)
            pre     = np.delete(pre,INDEX,0)
        mape = (sum(abs((pre - Y_test1)/(Y_test1)))/len(Y_test1))
        print('第{}个目标'.format(i))
        print('mse:',mse)
        print('rmse:',rmse)
        print('mae:',mae)
        print('mape:',mape)
        mse1.append(mse)
        rmse1.append(rmse)
        mae1.append(mae)
        mape1.append(mape)
    mse2.append(mse1)
    rmse2.append(rmse1)
    mae2.append(mae1)
    mape2.append(mape1)
    print('-----------------------------------')
mse2 = np.array(mse2)
rmse2 = np.array(rmse2)
mae2 = np.array(mae2)
mape2 = np.array(mape2)
print('交叉验证评估指标均值：')
for i in range(2):
    print('第{}个目标'.format(i))
    print('mse:',mse2[:,i].mean())
    print('rmse:',rmse2[:,i].mean())
    print('mae:',mae2[:,i].mean())
    print('mape:',mape2[:,i].mean())

交叉验证第1折
torch.Size([1450, 10346]) torch.Size([1450, 2]) torch.Size([363, 10346]) torch.Size([363, 2])
第0.8606402277946472个目标
mse: 1.0214497
rmse: 1.0106679
mae: 0.8199215
mape: 1.2806649540676722
第-0.984201967716217个目标
mse: 1.1213747
rmse: 1.0589498
mae: 0.83237535
mape: 1.4128187764987616
-----------------------------------
交叉验证第2折
torch.Size([1450, 10346]) torch.Size([1450, 2]) torch.Size([363, 10346]) torch.Size([363, 2])
第-0.9614611864089966个目标
mse: 0.93308234
rmse: 0.9659619
mae: 0.7537033
mape: 1.4593440854957855
第1.5767625570297241个目标
mse: 1.0112035
rmse: 1.0055861
mae: 0.815878
mape: 2.905199315429719
-----------------------------------
交叉验证第3折
torch.Size([1450, 10346]) torch.Size([1450, 2]) torch.Size([363, 10346]) torch.Size([363, 2])
第1.5256472826004028个目标
mse: 0.9925244
rmse: 0.99625516
mae: 0.7864683
mape: 1.3432833535530022
第-1.958536148071289个目标
mse: 0.91796017
rmse: 0.9581024
mae: 0.75851536
mape: 3.2951226473552464
-----------------------------------
交叉验证第4折
torch.Size